In [ ]:
import os
import sys
import difflib
import collections
import re 

# Import from pds_parser (ensure all node types are listed)
from pds_parser import (
    PdsParser, PdsBlock, PdsKeyValuePair, PdsList, 
    PdsComment, PdsBlankLine, PdsOperatorCondition, PdsNode
)
from pds_differ import PdsDiffer, PdsChange

print(f"--- Successfully imported PdsParser from: {PdsParser.__module__}.py ---")
print(f"--- Successfully imported PdsDiffer from: {PdsDiffer.__module__}.py ---")
print("-" * 80)

# --- Paths (UNCHANGED) ---
SIEGE_EVENTS_MOD_PATH = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes\events\siege_events.txt"
SIEGE_EVENTS_OLD_VANILLA_PATH = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver\events\siege_events.txt"
SIEGE_EVENTS_NEW_VANILLA_PATH = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game\events\siege_events.txt"
INNOVATIONS_MOD_PATH = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes\common\culture\innovations\00_tribal_innovations.txt"
INNOVATIONS_OLD_VANILLA_PATH = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver\common\culture\innovations\00_tribal_innovations.txt"
INNOVATIONS_NEW_VANILLA_PATH = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game\common\culture\innovations\00_tribal_innovations.txt"

# --- File I/O Helpers (UNCHANGED) ---
def get_file_content(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8-sig') as f: return f.read()
    except UnicodeDecodeError:
        try:
            with open(filepath, 'r', encoding='utf-8') as f: return f.read()
        except Exception as e_inner: sys.stderr.write(f"ERROR reading {filepath} (fallback): {e_inner}\n"); return None
    except FileNotFoundError: sys.stderr.write(f"WARNING: File not found: {filepath}\n"); return None
    except Exception as e: sys.stderr.write(f"ERROR reading {filepath}: {e}\n"); return None

def write_to_file(filepath, content):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    try:
        with open(filepath, 'w', encoding='utf-8-sig') as f: f.write(content); return True
    except Exception as e: sys.stderr.write(f"ERROR writing to {filepath}: {e}\n"); return False

# --- Tree Navigation Helpers (Use the versions from the previous full script response) ---
def _parse_indexed_identifier_for_nav(identifier_full):
    if isinstance(identifier_full, str) and "___" in identifier_full:
        parts = identifier_full.split("___", 1); base_key = parts[0]
        try: index = int(parts[1]); return base_key, index, True
        except ValueError: return identifier_full, 0, False
    return str(identifier_full) if identifier_full is not None else None, 0, False

def _get_key_from_path_segment(segment_str: str) -> str:
    if "___" in segment_str: return segment_str.split("___", 1)[0]
    return segment_str

def find_container_by_key_path(root_items, key_name_path: list[str]):
    current_container = root_items
    for i, key_name in enumerate(key_name_path):
        found_next = None
        search_in = current_container if isinstance(current_container, list) else (current_container.children if isinstance(current_container, PdsBlock) else [])
        for item in search_in:
            if isinstance(item, PdsBlock) and item.key == key_name: found_next = item; break
            elif isinstance(item, PdsKeyValuePair) and item.key == key_name and isinstance(item.value, PdsBlock): found_next = item.value; break
        if not found_next: return None
        current_container = found_next
        if i < len(key_name_path) - 1 and not isinstance(current_container, PdsBlock): return None 
    return current_container if isinstance(current_container, (PdsBlock, list)) else None

def find_node_by_old_path_in_sim_tree(sim_root_list, old_path_list: list[str]):
    current_level_nodes = sim_root_list; target_node = None
    for seg_idx, seg_str in enumerate(old_path_list):
        if "___INSERTED_at_" in seg_str and seg_idx == len(old_path_list) -1 : return None
        key_part, occ_idx, is_idxed = _parse_indexed_identifier_for_nav(seg_str)
        key_match_count = 0; found_lvl = False
        search_list = current_level_nodes if isinstance(current_level_nodes, list) else (current_level_nodes.children if isinstance(current_level_nodes, PdsBlock) else [])
        for node_sim in search_list:
            sim_key_str=None
            if hasattr(node_sim,'key') and node_sim.key is not None: sim_key_str=str(node_sim.key).replace("___","__^__").replace(".","^")
            elif isinstance(node_sim,PdsComment): sim_key_str=f"__COMMENT_{hash(node_sim.comment_text[:20])}"
            elif isinstance(node_sim,PdsBlankLine): sim_key_str=f"__BLANK_LINE_{node_sim.line_number}" # PdsBlankLine is imported
            else: sim_key_str=f"__{node_sim.__class__.__name__.upper()}_L{node_sim.line_number}"
            if sim_key_str == key_part:
                if not is_idxed or key_match_count == occ_idx: target_node=node_sim; found_lvl=True; break
                key_match_count +=1
        if not found_lvl: return None
        if seg_idx < len(old_path_list)-1:
            if isinstance(target_node, PdsBlock): current_level_nodes = target_node
            else: return None
    return target_node

def find_copied_node_recursive(sim_items: list, orig_new_node: PdsNode):
    if orig_new_node is None: return None, None
    q = collections.deque()
    for item in sim_items: q.append((item, sim_items))
    while q:
        curr, parent = q.popleft()
        if curr == orig_new_node: return curr, parent
        if isinstance(curr, PdsBlock):
            for child in curr.children: q.append((child, curr))
        elif isinstance(curr, (PdsKeyValuePair, PdsOperatorCondition)) and isinstance(curr.value, PdsBlock):
            for child_val_block in curr.value.children: q.append((child_val_block, curr.value))
    return None, None

def find_node_and_parent_in_sim_tree(sim_root_list, chg_obj: PdsChange):
    target_node, parent_node = None, None
    if chg_obj.type == 'MOD_ADDED':
        parent_keys = [_get_key_from_path_segment(s) for s in chg_obj.context_parent_path]
        if not parent_keys: parent_node = sim_root_list
        else: parent_node = find_container_by_key_path(sim_root_list, parent_keys)
        return None, parent_node

    if chg_obj.new_node is not None:
        target_node, parent_node = find_copied_node_recursive(sim_root_list, chg_obj.new_node)
        if not target_node:
            target_node = find_node_by_old_path_in_sim_tree(sim_root_list, chg_obj.key_path)
            if target_node:
                parent_path_keys = chg_obj.context_parent_path
                if not parent_path_keys: parent_node = sim_root_list
                else: parent_node = find_node_by_old_path_in_sim_tree(sim_root_list, parent_path_keys)
                if parent_node and isinstance(parent_node, list) and target_node not in parent_node: parent_node = None
                elif parent_node and isinstance(parent_node, PdsBlock) and target_node not in parent_node.children: parent_node = None
                if not parent_node: target_node = None
    elif chg_obj.old_node is not None: 
        target_node = find_node_by_old_path_in_sim_tree(sim_root_list, chg_obj.key_path)
        parent_path_keys = chg_obj.context_parent_path
        if not parent_path_keys: parent_node = sim_root_list
        else: parent_node = find_node_by_old_path_in_sim_tree(sim_root_list, parent_path_keys)
        if parent_node and target_node:
             if isinstance(parent_node, list) and target_node not in parent_node: parent_node = None
             elif isinstance(parent_node, PdsBlock) and target_node not in parent_node.children: parent_node = None
    
    if parent_node is not None and not isinstance(parent_node, (list, PdsBlock)): parent_node = None
    return target_node, parent_node

def normalize_for_comparison(text):
    if not text: return ""
    return re.sub(r'\n(\s*\n)+', '\n', text).strip()

g_subsumed_mod_block_paths = set()

def run_and_print_diff(test_name, old_filepath, mod_filepath, new_filepath):
    global g_subsumed_mod_block_paths
    g_subsumed_mod_block_paths = set() 

    print(f"\n{'='*20} RUNNING TEST: {test_name} {'='*20}")
    test_output_dir = os.path.join(os.getcwd(), "test_output", test_name.replace(" ", "_").replace("(", "").replace(")", ""))
    os.makedirs(test_output_dir, exist_ok=True)
    print(f"Output files for this test will be saved to: {test_output_dir}")

    old_content_raw = get_file_content(old_filepath)
    mod_content_raw = get_file_content(mod_filepath)
    new_content_raw = get_file_content(new_filepath)
    write_to_file(os.path.join(test_output_dir, os.path.basename(old_filepath).replace(".txt", "_OLD_RAW.txt")), old_content_raw or "")
    write_to_file(os.path.join(test_output_dir, os.path.basename(mod_filepath).replace(".txt", "_MOD_RAW.txt")), mod_content_raw or "")
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_NEW_RAW.txt")), new_content_raw or "")
    print(f"  Raw files saved.")

    parser = PdsParser() 
    print(f"  Parsing Old file: {os.path.basename(old_filepath)}")
    old_nodes = parser.parse_file(old_filepath)
    print(f"  Parsing Mod file: {os.path.basename(mod_filepath)}")
    mod_nodes = parser.parse_file(mod_filepath)
    print(f"  Parsing New file: {os.path.basename(new_filepath)}")
    new_nodes = parser.parse_file(new_filepath)

    if old_nodes is None: old_nodes = []
    if mod_nodes is None: mod_nodes = []
    if new_nodes is None: new_nodes = []
    if not (old_nodes or mod_nodes or new_nodes) and not (old_content_raw or mod_content_raw or new_content_raw):
        print(f"SKIPPING: No content for test '{test_name}'."); return
    
    reconstructed_old = PdsParser._nodes_to_string(old_nodes)
    reconstructed_mod = PdsParser._nodes_to_string(mod_nodes)
    reconstructed_new = PdsParser._nodes_to_string(new_nodes)
    write_to_file(os.path.join(test_output_dir, os.path.basename(old_filepath).replace(".txt", "_OLD_RECONSTRUCTED.txt")), reconstructed_old)
    write_to_file(os.path.join(test_output_dir, os.path.basename(mod_filepath).replace(".txt", "_MOD_RECONSTRUCTED.txt")), reconstructed_mod)
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_NEW_RECONSTRUCTED.txt")), reconstructed_new)
    print(f"  Reconstructed files saved.")
    
    normalized_old_raw = normalize_for_comparison(old_content_raw or "")
    normalized_mod_raw = normalize_for_comparison(mod_content_raw or "")
    normalized_new_raw = normalize_for_comparison(new_content_raw or "")

    if normalize_for_comparison(reconstructed_old) != normalized_old_raw: print(f"\nWARNING: Normalized reconstruction mismatch for OLD file: {os.path.basename(old_filepath)}.")
    if normalize_for_comparison(reconstructed_mod) != normalized_mod_raw: print(f"\nWARNING: Normalized reconstruction mismatch for MOD file: {os.path.basename(mod_filepath)}.")
    if normalize_for_comparison(reconstructed_new) != normalized_new_raw: print(f"\nWARNING: Normalized reconstruction mismatch for NEW file: {os.path.basename(new_filepath)}.")

    differ = PdsDiffer()
    changes = differ.diff_nodes(old_nodes, mod_nodes, new_nodes)
    print(f"\n--- DETECTED CHANGES for '{test_name}' ({len(changes)} changes) ---")
    if not changes: print("    No significant changes detected by differ.")
    for change in changes: print(change)

    simulated_merged_nodes_root_list = [node.copy() for node in new_nodes]

    def _simulate_apply_change(sim_tree_root_list, chg_obj: PdsChange): # chg_obj was correct
        global g_subsumed_mod_block_paths
        current_path_tuple = tuple(chg_obj.key_path)

        if chg_obj.type.startswith("MOD_"):
            for subsuming_path_tuple_prefix in g_subsumed_mod_block_paths:
                if len(current_path_tuple) > len(subsuming_path_tuple_prefix) and \
                   current_path_tuple[:len(subsuming_path_tuple_prefix)] == subsuming_path_tuple_prefix:
                    print(f"  Skipping subsumed MOD change: {chg_obj.type} at {'->'.join(chg_obj.key_path)} (subsumed by {'->'.join(subsuming_path_tuple_prefix)})")
                    return
        
        print(f"\n  Attempting to apply change: {chg_obj.type} at OldPath {'->'.join(chg_obj.key_path)}")
        sim_comment_text = f"SimMerge:{chg_obj.type}"

        def _add_comment(node, comment):
            if node and hasattr(node, 'comment_text_on_line'):
                node.comment_text_on_line = f"{node.comment_text_on_line} {comment}" if node.comment_text_on_line else comment
            elif node and isinstance(node, PdsBlock): node.children.insert(0, PdsComment(comment))

        target_node_in_sim, parent_in_sim = find_node_and_parent_in_sim_tree(sim_tree_root_list, chg_obj)
        debug_path_str = '->'.join(chg_obj.key_path) if chg_obj.key_path else "ROOT"

        if chg_obj.type not in ['MOD_ADDED', 'CONFLICT_ADDITION'] and not chg_obj.type.endswith("_DELETED_MOD_MODIFIED") and \
           target_node_in_sim is None and not (chg_obj.type.endswith("_DELETED") or chg_obj.type.endswith("_ALSO_DELETED")):
            print(f"  SIM MERGE FAIL ({chg_obj.type}): Target node not found. Path: '{debug_path_str}'. Skipping.")
            return
        if parent_in_sim is None and not (chg_obj.type.endswith("_DELETED") and target_node_in_sim is None):
            print(f"  SIM MERGE FAIL ({chg_obj.type}): Parent container not found. Path: '{debug_path_str}'. Skipping.")
            return
        print(f"    Path: {debug_path_str}, Parent Found: {parent_in_sim is not None} (Type: {type(parent_in_sim).__name__}), TargetInSim Found: {target_node_in_sim is not None}")

        if chg_obj.type == 'MOD_MODIFIED':
            if target_node_in_sim and parent_in_sim and chg_obj.mod_node:
                mod_node_copy = chg_obj.mod_node.copy(); _add_comment(mod_node_copy, sim_comment_text)
                replaced = False
                if isinstance(parent_in_sim, list):
                    try: idx = parent_in_sim.index(target_node_in_sim); parent_in_sim[idx] = mod_node_copy; replaced = True
                    except ValueError: pass
                elif isinstance(parent_in_sim, PdsBlock): replaced = parent_in_sim.replace_child(target_node_in_sim, mod_node_copy)
                if replaced:
                    print(f"      Applied MOD_MODIFIED. Path: {debug_path_str}")
                    if isinstance(mod_node_copy, PdsBlock): g_subsumed_mod_block_paths.add(current_path_tuple)
                else: print(f"    SIM MERGE ERROR (MOD_MODIFIED): Replacement failed. Path: {debug_path_str}")
            else: print(f"    SIM MERGE FAIL (MOD_MODIFIED): Missing elements. Path: {debug_path_str}")

        elif chg_obj.type == 'MOD_ADDED':
            if parent_in_sim and chg_obj.mod_node:
                mod_node_copy = chg_obj.mod_node.copy(); _add_comment(mod_node_copy, sim_comment_text)
                if isinstance(parent_in_sim, list): parent_in_sim.append(mod_node_copy)
                elif isinstance(parent_in_sim, PdsBlock): parent_in_sim.add_child_at_appropriate_location(mod_node_copy)
                else: print(f"    SIM MERGE ERROR (MOD_ADDED): Invalid parent type. Path: {debug_path_str}"); return
                print(f"      Applied MOD_ADDED. Path: {debug_path_str}")
            else: print(f"    SIM MERGE FAIL (MOD_ADDED): Missing parent or mod_node. Path: {debug_path_str}")
        
        elif chg_obj.type == 'MOD_DELETED':
            if target_node_in_sim and parent_in_sim:
                removed = False
                if isinstance(parent_in_sim, list):
                    try: parent_in_sim.remove(target_node_in_sim); removed = True
                    except ValueError: pass
                elif isinstance(parent_in_sim, PdsBlock): removed = parent_in_sim.remove_child(target_node_in_sim)
                if removed: print(f"      Applied MOD_DELETED. Path: {debug_path_str}")
                else: print(f"    SIM MERGE ERROR (MOD_DELETED): Removal failed. Path: {debug_path_str}")
            elif not target_node_in_sim: print(f"      MOD_DELETED: Target already absent. Path: {debug_path_str}"); _add_comment(parent_in_sim, sim_comment_text + " (target absent)")
            else: print(f"    SIM MERGE FAIL (MOD_DELETED): Missing parent. Path: {debug_path_str}")

        elif chg_obj.type in ('CONVERGED_MODIFICATION', 'MOD_ADDED_CONVERGED', 'VANILLA_MODIFIED', 'VANILLA_ADDED'):
            if target_node_in_sim: _add_comment(target_node_in_sim, sim_comment_text); print(f"      Commented for {chg_obj.type}. Path: {debug_path_str}")
            else: print(f"    SIM MERGE WARN ({chg_obj.type}): Target node not found to comment. Path: {debug_path_str}")

        # **FIXED TYPO HERE**
        elif chg_obj.type in ('VANILLA_DELETED', 'MOD_DELETED_VANILLA_ALSO_DELETED'): 
            print(f"      Noted {chg_obj.type} (node absent). Path: {debug_path_str}"); _add_comment(parent_in_sim, sim_comment_text + " (target absent)")
            
        elif chg_obj.type.startswith('CONFLICT'):
            if chg_obj.type == 'CONFLICT_MODIFIED':
                if target_node_in_sim and parent_in_sim and chg_obj.mod_node:
                    mod_node_copy = chg_obj.mod_node.copy(); _add_comment(mod_node_copy, sim_comment_text + "_MOD_CHOSEN")
                    replaced = False
                    if isinstance(parent_in_sim, list):
                        try: idx=parent_in_sim.index(target_node_in_sim);parent_in_sim[idx]=mod_node_copy;replaced=True
                        except ValueError:pass
                    elif isinstance(parent_in_sim, PdsBlock): replaced = parent_in_sim.replace_child(target_node_in_sim, mod_node_copy)
                    if replaced:
                        print(f"        Resolved {chg_obj.type}: Replaced with Mod. Path: {debug_path_str}")
                        if isinstance(mod_node_copy, PdsBlock): g_subsumed_mod_block_paths.add(current_path_tuple)
                    else: print(f"      SIM MERGE CONFLICT ERROR ({chg_obj.type}): Replacement failed. Path: {debug_path_str}")
                else: print(f"      SIM MERGE CONFLICT FAIL ({chg_obj.type}): Missing elements. Path: {debug_path_str}")
            elif chg_obj.type == 'CONFLICT_ADDITION':
                if parent_in_sim and chg_obj.mod_node:
                    mod_node_copy = chg_obj.mod_node.copy(); _add_comment(mod_node_copy, sim_comment_text + "_MOD_CHOSEN")
                    added_or_replaced = False
                    if target_node_in_sim: 
                        if isinstance(parent_in_sim, list):
                            try: idx=parent_in_sim.index(target_node_in_sim);parent_in_sim[idx]=mod_node_copy;added_or_replaced=True
                            except ValueError: parent_in_sim.append(mod_node_copy); added_or_replaced=True 
                        elif isinstance(parent_in_sim, PdsBlock):
                            if parent_in_sim.replace_child(target_node_in_sim, mod_node_copy): added_or_replaced=True
                            else: parent_in_sim.add_child_at_appropriate_location(mod_node_copy); added_or_replaced=True
                    else: 
                        if isinstance(parent_in_sim, list): parent_in_sim.append(mod_node_copy); added_or_replaced=True
                        elif isinstance(parent_in_sim, PdsBlock): parent_in_sim.add_child_at_appropriate_location(mod_node_copy); added_or_replaced=True
                    if added_or_replaced: print(f"        Resolved {chg_obj.type}: Applied Mod's addition. Path: {debug_path_str}")
                    else: print(f"      SIM MERGE CONFLICT ERROR ({chg_obj.type}): Add/Replace failed. Path: {debug_path_str}")
                else: print(f"      SIM MERGE CONFLICT FAIL ({chg_obj.type}): Missing elements. Path: {debug_path_str}")
            elif chg_obj.type == 'CONFLICT_DELETION_MOD_DELETED_VANILLA_MODIFIED':
                if target_node_in_sim: _add_comment(target_node_in_sim, sim_comment_text + "_VANILLA_MOD_KEPT"); print(f"        Resolved {chg_obj.type}: Kept Vanilla's. Path: {debug_path_str}")
                else: print(f"      SIM MERGE CONFLICT WARN ({chg_obj.type}): Vanilla's modified node not found. Path: {debug_path_str}")
            elif chg_obj.type == 'CONFLICT_DELETION_VANILLA_DELETED_MOD_MODIFIED':
                if parent_in_sim and chg_obj.mod_node:
                    mod_node_copy = chg_obj.mod_node.copy(); _add_comment(mod_node_copy, sim_comment_text + "_MOD_MOD_APPLIED")
                    if isinstance(parent_in_sim, list): parent_in_sim.append(mod_node_copy)
                    elif isinstance(parent_in_sim, PdsBlock): parent_in_sim.add_child_at_appropriate_location(mod_node_copy)
                    print(f"        Resolved {chg_obj.type}: Applied Mod's version (as add). Path: {debug_path_str}")
                    if isinstance(mod_node_copy, PdsBlock): g_subsumed_mod_block_paths.add(current_path_tuple)
                else: print(f"      SIM MERGE CONFLICT FAIL ({chg_obj.type}): Missing elements. Path: {debug_path_str}")
            else: print(f"    SIM MERGE WARN: Unhandled conflict type logic: {chg_obj.type}. Path: {debug_path_str}")
        else: print(f"    SIM MERGE NOTE: Change type '{chg_obj.type}' not requiring action or already handled. Path: {debug_path_str}")

    print(f"\n--- SIMULATING MERGE for '{test_name}' ---")
    for change_item in changes:
        _simulate_apply_change(simulated_merged_nodes_root_list, change_item)

    simulated_merged_content = PdsParser._nodes_to_string(simulated_merged_nodes_root_list)
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_SIMULATED_MERGED.txt")), simulated_merged_content)
    print(f"\n  Simulated merged content saved.")
    print(f"\n--- DIFF: SIMULATED MERGED vs NORMALIZED NEW VANILLA RAW for '{test_name}' ---")
    diff_filename = os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_MERGED_VS_NEW_RAW.diff"))
    normalized_simulated_merged = normalize_for_comparison(simulated_merged_content)
    with open(diff_filename, 'w', encoding='utf-8') as f_diff:
        diff_lines = list(difflib.unified_diff(
            normalized_new_raw.splitlines(keepends=True),
            normalized_simulated_merged.splitlines(keepends=True),
            fromfile='NORMALIZED_NEW_VANILLA_RAW', tofile='NORMALIZED_SIMULATED_MERGED', lineterm=''))
        if diff_lines: f_diff.writelines(diff_lines); print(f"  Diff (Normalized) saved to: {os.path.basename(diff_filename)}")
        else: print("  NORMALIZED SIMULATED MERGED is identical to NORMALIZED NEW VANILLA RAW.")
    print("-" * 80)

# --- Run Tests ---
print("Running diff tests with real CK3 files.")
# run_and_print_diff("Siege Events (real files)", SIEGE_EVENTS_OLD_VANILLA_PATH, SIEGE_EVENTS_MOD_PATH, SIEGE_EVENTS_NEW_VANILLA_PATH)
run_and_print_diff("00_tribal_innovations (real files)", INNOVATIONS_OLD_VANILLA_PATH, INNOVATIONS_MOD_PATH, INNOVATIONS_NEW_VANILLA_PATH)
print("\n--- All Tests Complete ---")

--- Successfully imported PdsParser from: pds_parser.py ---
--- Successfully imported PdsDiffer from: pds_differ.py ---
--------------------------------------------------------------------------------
Running diff tests with real CK3 files.

==================== RUNNING TEST: 00_tribal_innovations (real files) ====================
Output files for this test will be saved to: c:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\test_output\00_tribal_innovations_real_files
  Raw files saved.
  Parsing Old file: 00_tribal_innovations.txt
  Parsing Mod file: 00_tribal_innovations.txt
  Parsing New file: 00_tribal_innovations.txt
  Reconstructed files saved.


NameError: name 're' is not defined